# 4. Validação

Avaliação do modelo treinado sobre o split de teste. Caso não exista um run de treino com o `EXPERIMENT_ID` informado, o modelo base é avaliado.

## Preparação

In [ ]:
import sys
sys.path.append("..")
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from src import config, datasets, eval as eval_mod

## Configuração

In [ ]:
EXPERIMENT_ID        = "yolo-direct-all-datasets"  # identifica os pesos do treino
N_SAMPLE_PREDICTIONS = 10
DATASET_STAGE        = "processed"
WEIGHTS              = ""  # deixar vazio para carregar automaticamente do run de treino

# datasets de avaliação — independentes dos usados no treino
EVAL_DATASETS = datasets.available()
# opções: ["crowdhuman", "inside-bus-view", "passenger-deakin", "passenger-detection-bus"]

## Datasets

In [ ]:
data_yaml = datasets.prepare(EVAL_DATASETS, stage=DATASET_STAGE)

for name in EVAL_DATASETS:
    for split in ["valid", "test"]:
        img_dir = config.PROCESSED_DIR / name / split / "images"
        if img_dir.exists():
            n = len(list(img_dir.glob("*.*")))
            print(f"  {name}/{split}: {n} imagens")

## Pesos

In [ ]:
run_dir, weights_path = eval_mod.find_run(EXPERIMENT_ID)
weights_path = WEIGHTS or weights_path

print("Run dir:", run_dir)
print("Pesos:  ", weights_path)

## Modelo

In [ ]:
from ultralytics import YOLO
model = YOLO(str(weights_path))

## Avaliação

In [ ]:
metrics = eval_mod.run(EXPERIMENT_ID, model, data_yaml,
                       n_samples=N_SAMPLE_PREDICTIONS, run_dir=run_dir)
metrics